In [2]:
#Basic setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.linear_model as lm
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#Show all columns, without hiding any
pd.set_option('display.max_columns', None)

#Set the chart background to white with a grid for easier observation and comparison of data values.
sns.set(style="whitegrid")

In [3]:
#Load the dataset
path = "../Raw data/spotify_data clean.csv"
df = pd.read_csv(path)

#Quick look at the data
df.head()

,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min
0,3EJS5LyekDim1Tf5rBFmZl,Trippy Mane (ft. Project Pat),4,0,True,Diplo,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.55
1,1oQW6G2ZiwMuHqlPpP27DB,OMG!,1,0,True,Yelawolf,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,OMG!,2025-10-31,1,single,3.07
2,7mdkjzoIYlf1rx9EtBpGmU,Hard 2 Find,1,4,True,Riff Raff,48,193302,NaN,3E3zEAL8gUYWaLYB9L7gbp,Hard 2 Find,2025-10-31,1,single,2.55
3,67rW0Zl7oB3qEpD5YWWE5w,Still Get Like That (ft. Project Pat & Starrah),8,30,True,Diplo,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.69
4,15xptTfRBrjsppW0INUZjf,ride me like a harley,2,0,True,Rumelis,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,come closer / ride me like a harley,2025-10-30,2,single,2.39


In [4]:
df_cleaned = df.copy()

print("Initial shape of df_cleaned:", df_cleaned.shape)
display(df_cleaned.head())

Initial shape of df_cleaned: (8582, 15)


,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min
0,3EJS5LyekDim1Tf5rBFmZl,Trippy Mane (ft. Project Pat),4,0,True,Diplo,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.55
1,1oQW6G2ZiwMuHqlPpP27DB,OMG!,1,0,True,Yelawolf,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,OMG!,2025-10-31,1,single,3.07
2,7mdkjzoIYlf1rx9EtBpGmU,Hard 2 Find,1,4,True,Riff Raff,48,193302,NaN,3E3zEAL8gUYWaLYB9L7gbp,Hard 2 Find,2025-10-31,1,single,2.55
3,67rW0Zl7oB3qEpD5YWWE5w,Still Get Like That (ft. Project Pat & Starrah),8,30,True,Diplo,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.69
4,15xptTfRBrjsppW0INUZjf,ride me like a harley,2,0,True,Rumelis,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,come closer / ride me like a harley,2025-10-30,2,single,2.39


In [5]:
# 1. REMOVE DUPLICATED ROWS

duplicate_count = df_cleaned.duplicated().sum()

df_cleaned = df_cleaned.drop_duplicates()

print(f"Removed {duplicate_count} duplicated rows.")
print("Dataset shape after removing duplicates:", df_cleaned.shape)

Removed 0 duplicated rows.
Dataset shape after removing duplicates: (8582, 15)


In [6]:
# 2. CONVERT RELEASE DATE

df_cleaned['album_release_date'] = pd.to_datetime(
    df_cleaned['album_release_date'],
    errors='coerce'
)

df_cleaned['release_year'] = df_cleaned['album_release_date'].dt.year
df_cleaned['release_month'] = df_cleaned['album_release_date'].dt.month

print("album_release_date converted to datetime.")
print("release_year and release_month created.")

display(df_cleaned[['album_release_date', 'release_year', 'release_month']].head())

album_release_date converted to datetime.
release_year and release_month created.


,album_release_date,release_year,release_month
0,2025-10-31,2025,10
1,2025-10-31,2025,10
2,2025-10-31,2025,10
3,2025-10-31,2025,10
4,2025-10-30,2025,10


In [7]:
# 3. CONVERT NUMERICAL COLUMNS

numeric_cols = [
    'track_popularity',
    'artist_popularity',
    'artist_followers',
    'album_total_tracks',
    'track_number',
    'track_duration_min'
]

for col in numeric_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

print("Numerical columns converted successfully.")

print("\nData types of numerical columns:")
print(df_cleaned[numeric_cols].dtypes)

Numerical columns converted successfully.

Data types of numerical columns:
track_popularity        int64
artist_popularity       int64
artist_followers        int64
album_total_tracks      int64
track_number            int64
track_duration_min    float64
dtype: object


In [8]:
# 4. HANDLE MISSING CATEGORICAL VALUES

df_cleaned['artist_genres'] = df_cleaned['artist_genres'].fillna('Unknown')
df_cleaned['album_type'] = df_cleaned['album_type'].fillna('Unknown')

print("Missing values in artist_genres and album_type filled with 'Unknown'.")

print("\nMissing values after handling categorical columns:")
print(df_cleaned[['artist_genres', 'album_type']].isnull().sum())

Missing values in artist_genres and album_type filled with 'Unknown'.

Missing values after handling categorical columns:
artist_genres    0
album_type       0
dtype: int64


In [9]:
# 5. DROP ROWS WITH MISSING IMPORTANT VALUES

required_cols = [
    'track_popularity',
    'artist_popularity',
    'artist_followers',
    'track_duration_min',
    'album_total_tracks',
    'track_number',
    'release_year',
    'release_month'
]

before_drop = df_cleaned.shape[0]

df_cleaned = df_cleaned.dropna(subset=required_cols)

after_drop = df_cleaned.shape[0]

print(f"Dropped {before_drop - after_drop} rows due to missing important values.")
print("Dataset shape after dropping missing values:", df_cleaned.shape)

Dropped 0 rows due to missing important values.
Dataset shape after dropping missing values: (8582, 17)


In [10]:
# 6. DROP UNNECESSARY COLUMNS

# Columns that are not needed for the prediction model
columns_to_drop = [
    # ID columns
    'track_id',
    'album_id',
    'artist_id',

    # Text/name columns
    'track_name',
    'album_name',
    'artist_name',

    # Raw date column
    'album_release_date',

    # Raw genre column
    'artist_genres',

    # Target-related column
    'popularity_level'
]

# Drop only columns that actually exist in the DataFrame to avoid KeyError
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]

df.drop(columns=existing_columns_to_drop, inplace=True)

print("Dropped unnecessary columns:")
print(existing_columns_to_drop)

print("\nRemaining columns:")
print(df.columns.tolist())

Dropped unnecessary columns:
['track_id', 'album_id', 'track_name', 'album_name', 'artist_name', 'album_release_date', 'artist_genres']

Remaining columns:
['track_number', 'track_popularity', 'explicit', 'artist_popularity', 'artist_followers', 'album_total_tracks', 'album_type', 'track_duration_min']


In [11]:
# 7. REMOVE INVALID POPULARITY VALUES

before_filter = df_cleaned.shape[0]

df_cleaned = df_cleaned[
    (df_cleaned['track_popularity'] >= 0) &
    (df_cleaned['track_popularity'] <= 100) &
    (df_cleaned['artist_popularity'] >= 0) &
    (df_cleaned['artist_popularity'] <= 100)
]

after_filter = df_cleaned.shape[0]

print(f"Removed {before_filter - after_filter} rows with invalid popularity values.")
print("Dataset shape after filtering popularity values:", df_cleaned.shape)

Removed 0 rows with invalid popularity values.
Dataset shape after filtering popularity values: (8582, 17)


In [12]:
# 8. REMOVE INVALID DURATION AND FOLLOWERS

before_filter = df_cleaned.shape[0]

df_cleaned = df_cleaned[
    (df_cleaned['track_duration_min'] > 0) &
    (df_cleaned['artist_followers'] >= 0)
]

after_filter = df_cleaned.shape[0]

print(f"Removed {before_filter - after_filter} rows with invalid duration or followers.")
print("Dataset shape after filtering duration and followers:", df_cleaned.shape)

Removed 0 rows with invalid duration or followers.
Dataset shape after filtering duration and followers: (8582, 17)


In [13]:
# 9. REMOVE INVALID ALBUM TRACK INFORMATION

before_filter = df_cleaned.shape[0]

df_cleaned = df_cleaned[
    (df_cleaned['album_total_tracks'] > 0) &
    (df_cleaned['track_number'] > 0) &
    (df_cleaned['track_number'] <= df_cleaned['album_total_tracks'])
]

after_filter = df_cleaned.shape[0]

print(f"Removed {before_filter - after_filter} rows with invalid album track information.")
print("Dataset shape after filtering album track information:", df_cleaned.shape)

Removed 0 rows with invalid album track information.
Dataset shape after filtering album track information: (8582, 17)


In [14]:
# 10. FINAL CHECK AFTER DATA CLEANING

print("Final dataset shape after cleaning:", df_cleaned.shape)

print("\nMissing values after cleaning:")
print(df_cleaned.isnull().sum())

print("\nDuplicated rows after cleaning:")
print(df_cleaned.duplicated().sum())

print("\nData types after cleaning:")
df_cleaned.info()

display(df_cleaned.head())

Final dataset shape after cleaning: (8582, 17)

Missing values after cleaning:
track_id              0
track_name            0
track_number          0
track_popularity      0
explicit              0
artist_name           3
artist_popularity     0
artist_followers      0
artist_genres         0
album_id              0
album_name            0
album_release_date    0
album_total_tracks    0
album_type            0
track_duration_min    0
release_year          0
release_month         0
dtype: int64

Duplicated rows after cleaning:
0

Data types after cleaning:
<class 'pandas.DataFrame'>
RangeIndex: 8582 entries, 0 to 8581
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   track_id            8582 non-null   str           
 1   track_name          8582 non-null   str           
 2   track_number        8582 non-null   int64         
 3   track_popularity    8582 non-null   int64         
 4   

,track_id,track_name,track_number,track_popularity,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,track_duration_min,release_year,release_month
0,3EJS5LyekDim1Tf5rBFmZl,Trippy Mane (ft. Project Pat),4,0,True,Diplo,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.55,2025,10
1,1oQW6G2ZiwMuHqlPpP27DB,OMG!,1,0,True,Yelawolf,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,OMG!,2025-10-31,1,single,3.07,2025,10
2,7mdkjzoIYlf1rx9EtBpGmU,Hard 2 Find,1,4,True,Riff Raff,48,193302,Unknown,3E3zEAL8gUYWaLYB9L7gbp,Hard 2 Find,2025-10-31,1,single,2.55,2025,10
3,67rW0Zl7oB3qEpD5YWWE5w,Still Get Like That (ft. Project Pat & Starrah),8,30,True,Diplo,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,"d00mscrvll, Vol. 1",2025-10-31,9,album,1.69,2025,10
4,15xptTfRBrjsppW0INUZjf,ride me like a harley,2,0,True,Rumelis,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,come closer / ride me like a harley,2025-10-30,2,single,2.39,2025,10


In [15]:
df_cleaned.to_csv(r'..\Data\data_cleaned.csv', index=False)

***BASELINE MODEL***

LOGISTIC REGRESSION BASELINE MODEL

In [ ]:
import pandas as pd

# =========================
# READ DATA
# =========================

df = pd.read_csv(r'..\Data\data_cleaned.csv')

# check shape
print(df.shape)

# =========================
# CREATE BINARY TARGET
# =========================

def hit_or_not(x):

    if x > 80:
        return 1   # Hit
    else:
        return 0   # Not Hit

df['target'] = df['track_popularity'].apply(hit_or_not)

# =========================
# SELECT NUMERIC COLUMNS
# =========================

numeric_df = df.select_dtypes(
    include=['int64', 'float64']
)

# =========================
# CREATE X and y
# =========================

X = numeric_df.drop(
    columns=['track_popularity']
)

y = df['target']

# =========================
# CHECK
# =========================

print(X.shape)
print(y.shape)

print("\nTarget Distribution:\n")

print(
    y.value_counts()
)

# =========================
# TRAIN TEST SPLIT
# =========================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# LOGISTIC REGRESSION MODEL
# =========================

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

# =========================
# TRAIN MODEL
# =========================

model.fit(X_train, y_train)

# =========================
# PREDICT
# =========================

y_pred = model.predict(X_test)

# =========================
# EVALUATE
# =========================

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

print("\nAccuracy:")

print(
    accuracy_score(y_test, y_pred)
)

print("\nClassification Report:\n")

print(
    classification_report(y_test, y_pred)
)

(8582, 17)
(8582, 8)
(8582,)

Target Distribution:

target
0    7990
1     592
Name: count, dtype: int64

Accuracy:
0.9312754804892254

Classification Report:

              precision    recall  f1-score   support

           0       0.93      1.00      0.96      1599
           1       0.00      0.00      0.00       118

    accuracy                           0.93      1717
   macro avg       0.47      0.50      0.48      1717
weighted avg       0.87      0.93      0.90      1717



C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_clas

RANDOM FOREST BASELINE MODEL

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# =====================================
# 1. CREATE BINARY TARGET
# =====================================

def hit_or_not(x):

    if x > 80:
        return "Hit"
    else:
        return "Not Hit"

df['target'] = df['track_popularity'].apply(hit_or_not)

# =====================================
# 2. CREATE X (FEATURES)
# =====================================

# select numeric columns
X = df.select_dtypes(include=['int64', 'float64'])

# drop the popularity column to avoid data leakage
X = X.drop(columns=['track_popularity'])

# =====================================
# 3. CREATE y (TARGET)
# =====================================

y = df['target']

# =====================================
# 4. TRAIN TEST SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================
# 5. RANDOM FOREST BASELINE MODEL
# =====================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# =====================================
# 6. TRAIN MODEL
# =====================================

rf_model.fit(X_train, y_train)

# =====================================
# 7. PREDICT
# =====================================

y_pred = rf_model.predict(X_test)

# =====================================
# 8. EVALUATE MODEL
# =====================================

print("Random Forest Accuracy:")

print(
    accuracy_score(y_test, y_pred)
)

print("\nClassification Report:\n")

print(
    classification_report(y_test, y_pred)
)

# =====================================
# 9. CHECK TARGET DISTRIBUTION
# =====================================

print("\nTarget Distribution:\n")

print(
    df['target'].value_counts()
)

Random Forest Accuracy:
0.930110658124636

Classification Report:

              precision    recall  f1-score   support

         Hit       0.46      0.09      0.15       118
     Not Hit       0.94      0.99      0.96      1599

    accuracy                           0.93      1717
   macro avg       0.70      0.54      0.56      1717
weighted avg       0.90      0.93      0.91      1717


Target Distribution:

target
Not Hit    7990
Hit         592
Name: count, dtype: int64


The first baseline model achieved a high accuracy of 93%, but failed to detect any “Hit” cases, resulting in a recall and F1-score of 0.00 for the minority class. This indicates that the model mainly predicted the majority class (“Not Hit”) due to the imbalanced dataset.

In comparison, the Random Forest model achieved a similar accuracy while being able to identify some “Hit” cases, with higher precision, recall, and F1-score for the minority class. Although the performance is still limited, Random Forest showed better capability in learning patterns from the target class.

Therefore, Random Forest was selected as the final model because it provided more balanced and meaningful classification performance.